# Climate profiles CDF interactive plots 
Author: Nicole Keeney<br>
Date: June 8, 2026

In [ ]:
import pandas as pd
import xarray as xr
import hvplot.pandas
import numpy as np
import hvplot.xarray
from importlib.resources import files
from dask.diagnostics import ProgressBar
import matplotlib.pyplot as plt

import climakitae as ck
from climakitae.core.constants import UNSET
from climakitae.core.data_export import write_tmy_file

# Disable zoom / active scrolling
# Fixes the plot in place, better for website :) 
import holoviews as hv
hv.plotting.bokeh.element.ElementPlot.active_tools = ['pan']

# Initialize ClimateData object
cd = ck.ClimateData(verbosity=-1)

## Set up 

In [ ]:
# read in station file of CA HadISD stations
stn_filepath_s3 = "s3://cadcat/hadisd/hadisd_stations.csv"
stn_file = pd.read_csv(stn_filepath_s3, index_col=[0])
stn_file.head(5)

# grab airport
stn_name = "Los Angeles International Airport (KLAX)"
stn_code = stn_file.loc[stn_file["station"] == stn_name]["station id"].item()
one_station = stn_file.loc[stn_file["station"] == stn_name]
stn_lat = one_station.LAT_Y.item()
stn_lon = one_station.LON_X.item()
stn_state = one_station.state.item()

# Output results
print("station coordinates:", (stn_lat, stn_lon))
print("station code:", stn_code)

# selected reference period
start_year = 2005
end_year = 2020

# selected models
data_models = [
    "wrf_ucla_taiesm1_ssp370_r1i1p1f1",
    "wrf_ucla_ec-earth3_ssp370_r1i1p1f1",
    "wrf_ucla_miroc6_ssp370_r1i1p1f1",
    "wrf_ucla_mpi-esm1-2-hr_ssp370_r3i1p1f1",
]

# map variable names to descriptive names and units
var_info = {
    "t2": {"long_name": "Air temperature at 2m", "units": "degC"},
}

## Download data

In [ ]:
t2_ds = (
    cd.catalog("cadcat")
    .variable("t2")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("1hr")
    .grid_label("d03")
    .experiment_id(["historical", "ssp370"])
    .processes(
        {
            "convert_units": "degC",
            "time_slice": (start_year, end_year + 1),
            "clip": (stn_lat, stn_lon),
            "convert_to_local_time": {"convert": "yes"},
        }
    )
    .get()
)

# Subset simulations and convert to DataArray
t2_ds = t2_ds.sel(sim=data_models, time=slice(str(start_year), str(end_year))).t2

# Load into memory using dask progress bar
with ProgressBar():
    t2_ds.load()

# max air temp
max_airtemp_data = t2_ds.resample(time="1D").max()  # daily max air temp
max_airtemp_data.name = "Daily max air temperature"  # rename for clarity

# min air temp
min_airtemp_data = t2_ds.resample(time="1D").min()  # daily min air temp
min_airtemp_data.name = "Daily min air temperature"  # rename for clarity

# mean air temp
mean_airtemp_data = t2_ds.resample(time="1D").mean()  # daily mean air temp
mean_airtemp_data.name = "Daily mean air temperature"  # rename for clarity

# delete large DataArray from memory-- no longer needed
del t2_ds

In [ ]:
# wind speed
ws_ds = (
    cd.catalog("cadcat")
    .variable("wind_speed_10m")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("1hr")
    .grid_label("d03")
    .experiment_id(["historical", "ssp370"])
    .processes(
        {
            "time_slice": (start_year, end_year + 1),
            "clip": (stn_lat, stn_lon),
            "convert_to_local_time": {"convert": "yes"},
        }
    )
    .get()
)

# Subset simulations and convert to DataArray
ws_ds = ws_ds.sel(
    sim=data_models, time=slice(str(start_year), str(end_year))
).wind_speed_10m

# Load into memory using dask progress bar
with ProgressBar():
    ws_ds.load()

# max wind speed
max_windspd_data = ws_ds.resample(time="1D").max()  # daily max wind speed
max_windspd_data.name = "Daily max wind speed"  # rename for clarity

# mean wind speed
mean_windspd_data = ws_ds.resample(time="1D").mean()  # daily mean wind speed
mean_windspd_data.name = "Daily mean wind speed"  # rename for clarity

# delete large DataArray from memory-- no longer needed
del ws_ds

In [ ]:
# dew point temperature
dewpt_ds = (
    cd.catalog("cadcat")
    .variable("dew_point_2m")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("1hr")
    .grid_label("d03")
    .experiment_id(["historical", "ssp370"])
    .processes(
        {
            "time_slice": (start_year, end_year + 1),
            "clip": (stn_lat, stn_lon),
            "convert_units": "degC",
            "convert_to_local_time": {"convert": "yes"},
        }
    )
    .get()
)

# Subset simulations and convert to DataArray
dewpt_ds = dewpt_ds.sel(
    sim=data_models, time=slice(str(start_year), str(end_year))
).dew_point_2m

# Load into memory using dask progress bar
with ProgressBar():
    dewpt_ds.load()

# max dew point
max_dewpt_data = dewpt_ds.resample(time="1D").max()  # daily max dewpoint temp
max_dewpt_data.name = "Daily max dewpoint temperature"  # rename for clarity

# min dew point
min_dewpt_data = dewpt_ds.resample(time="1D").min()  # daily min dewpoint temp
min_dewpt_data.name = "Daily min dewpoint temperature"  # rename for clarity

# mean dew point
mean_dewpt_data = dewpt_ds.resample(time="1D").mean()  # daily mean dewpoint temp
mean_dewpt_data.name = "Daily mean dewpoint temperature"  # rename for clarity

# delete large DataArray from memory-- no longer needed
del dewpt_ds

In [ ]:
# global irradiance
global_irradiance_ds = (
    cd.catalog("cadcat")
    .variable("swdnb")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("1hr")
    .grid_label("d03")
    .experiment_id(["historical", "ssp370"])
    .processes(
        {
            "time_slice": (start_year, end_year + 1),
            "clip": (stn_lat, stn_lon),
            "convert_to_local_time": {"convert": "yes"},
        }
    )
    .get()
)

# Subset simulations and convert to DataArray
global_irradiance_ds = global_irradiance_ds.sel(
    sim=data_models, time=slice(str(start_year), str(end_year))
).swdnb

# Load into memory using dask progress bar
with ProgressBar():
    global_irradiance_ds.load()

# total global horizontal irradiance (accumulation of hourly data over a 24-hour period)
total_ghi_data = global_irradiance_ds.resample(time="1D").sum()
total_ghi_data.name = "Global horizontal irradiance"  # rename for clarity

# delete large DataArray from memory-- no longer needed
del global_irradiance_ds

In [ ]:
# direct normal irradiance
direct_irradiance_ds = (
    cd.catalog("cadcat")
    .variable("swddni")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("1hr")
    .grid_label("d03")
    .experiment_id(["historical", "ssp370"])
    .processes(
        {
            "time_slice": (start_year, end_year + 1),
            "clip": (stn_lat, stn_lon),
            "convert_to_local_time": {"convert": "yes"},
        }
    )
    .get()
)

# Subset simulations and convert to DataArray
direct_irradiance_ds = direct_irradiance_ds.sel(
    sim=data_models, time=slice(str(start_year), str(end_year))
).swddni

# Load into memory using dask progress bar
with ProgressBar():
    direct_irradiance_ds.load()

# total direct normal irradiance (accumulation of hourly data over a 24-hour period)
total_dni_data = direct_irradiance_ds.resample(time="1D").sum()
total_dni_data.name = "Direct normal irradiance"  # rename for clarity

# delete large DataArray from memory-- no longer needed
del direct_irradiance_ds

## Merge variables into a single data object

In [ ]:
all_vars = xr.merge(
    [
        max_airtemp_data,
        min_airtemp_data,
        mean_airtemp_data,
        max_dewpt_data,
        min_dewpt_data,
        mean_dewpt_data,
        max_windspd_data,
        mean_windspd_data,
        total_ghi_data,
        total_dni_data,
    ]
)

## Compute CDF 

In [ ]:
def compute_cdf(da):
    """Compute the cumulative density function for an input DataArray"""
    da_np = da.values  # Get numpy array of values
    num_samples = 1024  # Number of samples to generate
    count, bins_count = np.histogram(  # Create a numpy histogram of the values
        da_np,
        bins=np.linspace(
            da_np.min(),  # Start at the minimum value of the array
            da_np.max(),  # End at the maximum value of the array
            num_samples,
        ),
    )
    cdf_np = np.cumsum(count / sum(count))  # Compute the CDF

    # Turn the CDF array into xarray DataArray
    # New dimension is the bin values
    cdf_da = xr.DataArray(
        [bins_count[1:], cdf_np],
        dims=["data", "bin_number"],
        coords={
            "data": ["bins", "probability"],
        },
    )
    cdf_da.name = da.name
    return cdf_da


def get_cdf_by_sim(da):
    # Group the DataArray by simulation
    return da.groupby("sim").map(compute_cdf)


def get_cdf_by_mon_and_sim(da):
    # Group the DataArray by month in the year
    return da.groupby("time.month").map(get_cdf_by_sim)


def get_cdf(ds):
    """Get the cumulative density function.

    Parameters
    -----------
    ds: xr.Dataset
        Input data to compute CDF for
    Returns
    -------
    xr.Dataset
    """
    return ds.map(get_cdf_by_mon_and_sim)

In [ ]:
def get_cdf_monthly(ds):
    """Get the cumulative density function by unique mon-yr combos

    Parameters
    -----------
    ds: xr.Dataset
        Input data to compute CDF for
    Returns
    -------
    xr.Dataset
    """

    def get_cdf_mon_yr(da):
        return da.groupby("time.year").map(get_cdf_by_mon_and_sim)

    return ds.map(get_cdf_mon_yr)

In [ ]:
cdf_climatology = get_cdf(all_vars)

In [ ]:
cdf_monthly = get_cdf_monthly(all_vars)

# Remove the years for the Pinatubo eruption
cdf_monthly = cdf_monthly.where(
    (~cdf_monthly.year.isin([1991, 1992, 1993, 1994])), np.nan, drop=True
)

## Make plots 

In [ ]:
def plot_cdf(cdf_da: xr.DataArray, groupby="month") -> hvplot:
    """
    Plot CDF with simulations as a grouped overlay.

    Parameters
    ----------
    cdf_da : xr.DataArray
        Output of compute_cdf, with a 'data' dimension containing 'bins' and 'probability'.
    groupby: str, list 
        "month" or ["month","year"]

    Returns
    -------
    hvplot
        CDF plot object.
    """
    month_names = [
        "January", "February", "March", "April", "May", "June",
        "July", "August", "September", "October", "November", "December"
    ]
    ds = xr.merge(
        [
            cdf_da.sel(data="probability", drop=True).rename("probability"),
            cdf_da.sel(data="bins", drop=True).rename("bins"),
        ]
    )
    df = ds.to_dataframe().reset_index()
    df["month"] = df["month"].map(lambda x: month_names[x - 1])

    cdf_pl = df.hvplot(
        "bins",
        "probability",
        by="sim",
        groupby=groupby,
        width=650,
        height=400,
        widget_location="bottom",
        legend="bottom_right",
        grid=True,
        active_scroll=None,
        xlabel=f"{cdf_da.name} ({cdf_da.attrs['units']})",
        xlim=(df["bins"].min(), df["bins"].max()),
        ylabel="Probability (0-1)",
    )
    return cdf_pl

In [ ]:
# Choose your desired variable
var = "Daily max air temperature"

# Make plot
pl1 = plot_cdf(cdf_climatology[var], groupby="month")

# Save plot
hvplot.save(pl1, 'tmy_cdf_annual.html')

# Display in-notebook
pl1

In [ ]:
pl2 = plot_cdf(cdf_monthly[var], groupby=["month","year"])
hvplot.save(pl2, 'tmy_cdf_mon_yr_grouped.html')
pl2